# Explainable Hybrid Stacking Ensemble for Postpartum Anxiety Screening

**Dataset:** `postpartum_depression.csv` (1,503 survey responses, 9 questions + age)
**Goal:** predict the answer to *"Feeling anxious"* (Yes / No) from the other answers, compare seven models, and explain the best one with SHAP and LIME.

### How the notebook is organised

| Part | Tasks | What happens |
|---|---|---|
| A | 1–3 | Setup and loading |
| B | 4–10 | Understanding the data (EDA) |
| C | 11–13 | Preparing the data |
| D | 14–22 | The seven competing models (and why each one is used) |
| E | 23–25 | 10-fold cross-validation competition |
| F | 26–35 | Final test, figures and statistical tests |
| G | 36–49 | Explainable AI: SHAP, LIME and SHAP-vs-LIME comparison |
| H | 50–53 | Trustworthy-AI extras: conformal prediction, counterfactuals, fairness |
| I | 54–55 | Robustness check: duplicate-aware validation |
| J | 56–57 | Comparison with published work, final summary |

Every code cell does **one small task**. বাংলা নোট: প্রতিটি সেলের উপরে লেখা আছে সেলটি কী করে এবং কেন।

## Part A — Setup
### Task 1: Import the libraries
সব লাইব্রেরি এক জায়গায় ইমপোর্ট করা হলো।

In [ ]:
import time
import warnings
from itertools import combinations
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy import stats

from sklearn.model_selection import train_test_split, StratifiedKFold, StratifiedGroupKFold
from sklearn.model_selection import cross_val_score, cross_val_predict
from sklearn.preprocessing import OneHotEncoder, OrdinalEncoder
from sklearn.pipeline import make_pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, StackingClassifier
from sklearn.neural_network import MLPClassifier
from sklearn.calibration import calibration_curve
from sklearn.inspection import permutation_importance
from sklearn import metrics

from xgboost import XGBClassifier
from lightgbm import LGBMClassifier
from catboost import CatBoostClassifier
import optuna
import shap
from lime.lime_tabular import LimeTabularExplainer

warnings.filterwarnings("ignore")
optuna.logging.set_verbosity(optuna.logging.WARNING)

### Task 2: Settings
ফাইলের পাথ, টার্গেট কলাম, র‍্যান্ডম সিড আর রঙ — সব এক জায়গায়।

In [ ]:
DATA_FILE = "../data/raw/postpartum_depression.csv"
RESULTS = Path("../results")
RESULTS.mkdir(exist_ok=True)

TARGET = "Feeling anxious"
SEED = 42
HYBRID = "Hybrid Stack (proposed)"

BLUE, ORANGE, GREY, GREEN = "#2a78d6", "#eb6834", "#9aa0a6", "#2e9e6b"
plt.rcParams.update({"figure.figsize": (7, 4), "figure.dpi": 110, "axes.spines.top": False,
                     "axes.spines.right": False, "savefig.bbox": "tight"})

### Task 3: Load the dataset
CSV ফাইলটি শুধু পড়া হয়, কখনো বদলানো হয় না।

In [ ]:
raw = pd.read_csv(DATA_FILE)
print("Rows and columns:", raw.shape)
raw.head()

## Part B — Understanding the data
### Task 4: Column types and possible answers
প্রতিটি প্রশ্নে কী কী উত্তর আছে তা দেখা।

In [ ]:
for column in raw.columns[1:]:
    print(f"{column:45s} -> {sorted(raw[column].dropna().unique())}")

### Task 5: Missing answers
কোন প্রশ্নে কতগুলো উত্তর ফাঁকা আছে।

In [ ]:
missing = raw.isna().sum()
print(missing[missing > 0])
print("Total missing cells:", missing.sum())

### Task 6: Repeated answer patterns
অনেক উত্তরদাতা হুবহু একই উত্তর দিয়েছেন। এটা গুরুত্বপূর্ণ তথ্য, তাই শুরুতেই গোনা হলো।
This matters for validation: we use it again in **Part I**.

In [ ]:
answers = raw.drop(columns="Timestamp")
print("All responses           :", len(answers))
print("Unique answer patterns  :", len(answers.drop_duplicates()))
print("Repeated responses      :", answers.duplicated().sum())

### Task 7: Clean the data
* `Timestamp` is dropped (it is not a symptom).
* A missing answer becomes its own category, **"No answer"**. Nothing is learned from the data here, so there is no leakage.

In [ ]:
data = answers.fillna("No answer")
FEATURES = [c for c in data.columns if c != TARGET]
print("Features:", FEATURES)

### Task 8: Figure — class balance
কতজন anxious আর কতজন নয়।

In [ ]:
counts = data[TARGET].value_counts()
plt.bar(["Not anxious", "Anxious"], [counts["No"], counts["Yes"]], color=[GREY, BLUE])
for i, v in enumerate([counts["No"], counts["Yes"]]):
    plt.text(i, v + 10, f"{v} ({v / len(data):.0%})", ha="center")
plt.ylabel("Responses")
plt.title("Class balance of the target")
plt.savefig(RESULTS / "fig01_class_balance.png")
plt.show()

### Task 9: Figure — anxious rate for every answer
প্রতিটি প্রশ্নের প্রতিটি উত্তরে কত শতাংশ anxious।

In [ ]:
fig, axes = plt.subplots(3, 3, figsize=(13, 9))
for ax, feature in zip(axes.ravel(), FEATURES):
    rate = (data[TARGET] == "Yes").groupby(data[feature]).mean() * 100
    ax.bar(rate.index, rate.values, color=BLUE)
    ax.axhline((data[TARGET] == "Yes").mean() * 100, color=GREY, linestyle="--")
    ax.set_title(feature, fontsize=10)
    ax.set_ylabel("% anxious")
    ax.tick_params(axis="x", labelsize=8, rotation=15)
fig.suptitle("Anxious rate by answer (dashed line = overall rate)")
fig.tight_layout()
fig.savefig(RESULTS / "fig02_anxious_rate_by_answer.png")
plt.show()

### Task 10: Figure — strength of association with the target (Cramér's V)
Cramér's V মান 0 (কোনো সম্পর্ক নেই) থেকে 1 (পূর্ণ সম্পর্ক)। মডেল ছাড়াই বোঝা যায় কোন প্রশ্ন বেশি গুরুত্বপূর্ণ।

In [ ]:
def cramers_v(a, b):
    table = pd.crosstab(a, b)
    chi2 = stats.chi2_contingency(table)[0]
    return np.sqrt(chi2 / (table.values.sum() * (min(table.shape) - 1)))

association = pd.Series({f: cramers_v(data[f], data[TARGET]) for f in FEATURES}).sort_values()
plt.barh(association.index, association.values, color=BLUE)
plt.xlabel("Cramér's V with 'Feeling anxious'")
plt.title("Association of each question with the target")
plt.savefig(RESULTS / "fig03_cramers_v.png")
plt.show()

## Part C — Preparing the data
### Task 11: Turn answers into numbers
Each answer becomes an integer code (0, 1, 2 …). We keep **one column per question** so that SHAP and LIME can later talk about whole questions.

In [ ]:
encoder = OrdinalEncoder()
X = pd.DataFrame(encoder.fit_transform(data[FEATURES]).astype(int), columns=FEATURES)
y = (data[TARGET] == "Yes").astype(int).to_numpy()

ANSWERS = {f: list(c) for f, c in zip(FEATURES, encoder.categories_)}   # code -> answer text
X.head()

### Task 12: Train / test split (80% / 20%, stratified)
টেস্ট সেট একদম শেষ পর্যন্ত আলাদা রাখা হয়।

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.20, stratify=y, random_state=SEED)
print("Train:", X_train.shape, " Test:", X_test.shape)
print("Anxious share  train: %.3f  test: %.3f" % (y_train.mean(), y_test.mean()))

### Task 13: A small helper — one-hot encoding inside a pipeline
The integer codes have no real order, so every model first one-hot encodes them. Putting this inside the pipeline means it is re-learned on each training fold.

In [ ]:
def with_onehot(model):
    return make_pipeline(OneHotEncoder(handle_unknown="ignore", sparse_output=False), model)

models = {}        # every competing model is stored here

## Part D — The seven competing models

| # | Model | Family | Why it is in the study |
|---|---|---|---|
| 1 | Logistic Regression | Linear | Simple, transparent baseline. Shows what is possible **without** interactions |
| 2 | Random Forest | Bagging of trees | Classic strong baseline; low variance |
| 3 | XGBoost | Gradient boosting | The most used boosting model in health-ML papers |
| 4 | LightGBM | Gradient boosting (leaf-wise) | Fast, different tree-growing strategy |
| 5 | CatBoost | Ordered boosting | Designed for categorical data, like this survey |
| 6 | MLP | Neural network | Represents deep learning |
| 7 | **Hybrid Stack (proposed)** | Stacking | Combines the four different families; a meta-learner decides whom to trust |

### Task 14: Model 1 — Logistic Regression

In [ ]:
models["Logistic Regression"] = with_onehot(LogisticRegression(max_iter=2000))

### Task 15: Model 2 — Random Forest

In [ ]:
models["Random Forest"] = with_onehot(RandomForestClassifier(n_estimators=300, random_state=SEED))

### Task 16: Model 3 — XGBoost, tuned with Optuna
Optuna (Bayesian hyper-parameter search) tries 25 settings. Each one is scored with 3-fold cross-validation **on the training set only**.

In [ ]:
def xgb_objective(trial):
    model = with_onehot(XGBClassifier(
        n_estimators=trial.suggest_int("n_estimators", 100, 500),
        max_depth=trial.suggest_int("max_depth", 3, 10),
        learning_rate=trial.suggest_float("learning_rate", 0.02, 0.3, log=True),
        subsample=trial.suggest_float("subsample", 0.6, 1.0),
        colsample_bytree=trial.suggest_float("colsample_bytree", 0.6, 1.0),
        random_state=SEED))
    return cross_val_score(model, X_train, y_train, cv=3, scoring="accuracy").mean()

xgb_study = optuna.create_study(direction="maximize", sampler=optuna.samplers.TPESampler(seed=SEED))
xgb_study.optimize(xgb_objective, n_trials=25)
print("Best CV accuracy:", round(xgb_study.best_value, 4))
print("Best settings   :", xgb_study.best_params)

models["XGBoost"] = with_onehot(XGBClassifier(**xgb_study.best_params, random_state=SEED))

### Task 17: Model 4 — LightGBM, tuned with Optuna

In [ ]:
def lgbm_objective(trial):
    model = with_onehot(LGBMClassifier(
        n_estimators=trial.suggest_int("n_estimators", 100, 500),
        num_leaves=trial.suggest_int("num_leaves", 8, 64),
        learning_rate=trial.suggest_float("learning_rate", 0.02, 0.3, log=True),
        min_child_samples=trial.suggest_int("min_child_samples", 1, 20),
        colsample_bytree=trial.suggest_float("colsample_bytree", 0.6, 1.0),
        random_state=SEED, verbose=-1))
    return cross_val_score(model, X_train, y_train, cv=3, scoring="accuracy").mean()

lgbm_study = optuna.create_study(direction="maximize", sampler=optuna.samplers.TPESampler(seed=SEED))
lgbm_study.optimize(lgbm_objective, n_trials=25)
print("Best CV accuracy:", round(lgbm_study.best_value, 4))
print("Best settings   :", lgbm_study.best_params)

models["LightGBM"] = with_onehot(LGBMClassifier(**lgbm_study.best_params, random_state=SEED, verbose=-1))

### Task 18: Model 5 — CatBoost

In [ ]:
models["CatBoost"] = with_onehot(CatBoostClassifier(iterations=400, depth=6, learning_rate=0.1,
                                                    random_seed=SEED, verbose=0,
                                                    allow_writing_files=False))

### Task 19: Model 6 — MLP neural network

In [ ]:
models["MLP Neural Net"] = with_onehot(MLPClassifier(hidden_layer_sizes=(64, 32), max_iter=1000, random_state=SEED))

### Task 20: Model 7 — the proposed Hybrid Stack

```
            answers of one person
   ┌───────────┬───────────┬───────────┬───────────┐
 CatBoost    XGBoost   Random Forest    MLP          <- level 0: four different families
   └───────────┴─────┬─────┴───────────┘
        their out-of-fold probabilities
                     │
           Logistic Regression                       <- level 1: meta-learner
                     │
             final probability
```

* **Why stacking?** Each family makes different mistakes. The meta-learner learns how much to trust each one.
* **Why out-of-fold?** The meta-learner only sees predictions for people the base model did not train on (`cv=5`), so it cannot simply copy an overfitted model.

In [ ]:
base_models = [
    ("catboost", models["CatBoost"]),
    ("xgboost", models["XGBoost"]),
    ("forest", models["Random Forest"]),
    ("mlp", models["MLP Neural Net"]),
]
models[HYBRID] = StackingClassifier(estimators=base_models, final_estimator=LogisticRegression(max_iter=2000),
                                    stack_method="predict_proba", cv=5)

### Task 21: List of competitors

In [ ]:
for number, name in enumerate(models, start=1):
    print(number, name)

### Task 22: Figure — how Optuna improved over its trials
টিউনিং সত্যিই কাজ করেছে কিনা দেখা।

In [ ]:
for study, label, colour in [(xgb_study, "XGBoost", BLUE), (lgbm_study, "LightGBM", ORANGE)]:
    best_so_far = np.maximum.accumulate([t.value for t in study.trials])
    plt.plot(range(1, len(best_so_far) + 1), best_so_far * 100, label=label, color=colour)
plt.xlabel("Optuna trial")
plt.ylabel("Best 3-fold CV accuracy so far (%)")
plt.title("Hyper-parameter search progress")
plt.legend()
plt.savefig(RESULTS / "fig04_optuna_progress.png")
plt.show()

## Part E — Cross-validation competition
### Task 23: 10-fold stratified cross-validation on the training set
সব মডেল একই ১০টি ফোল্ডে পরীক্ষা দেয়, তাই তুলনা ন্যায্য। (কয়েক মিনিট লাগতে পারে।)

In [ ]:
folds = StratifiedKFold(n_splits=10, shuffle=True, random_state=SEED)
cv_scores = {}
for name, model in models.items():
    start = time.time()
    cv_scores[name] = cross_val_score(model, X_train, y_train, cv=folds, scoring="accuracy")
    print(f"{name:26s} {cv_scores[name].mean() * 100:.2f}%  ({time.time() - start:.0f}s)")
cv_scores = pd.DataFrame(cv_scores)

### Task 24: Cross-validation table (mean ± standard deviation)

In [ ]:
cv_table = pd.DataFrame({"cv_accuracy_mean_%": cv_scores.mean() * 100,
                         "cv_accuracy_std_%": cv_scores.std() * 100}).round(2)
cv_table = cv_table.sort_values("cv_accuracy_mean_%", ascending=False)
cv_table.to_csv(RESULTS / "cv_results.csv")
cv_table

### Task 25: Figure — cross-validation accuracy of every model

In [ ]:
order = cv_table.index[::-1]
plt.boxplot([cv_scores[m] * 100 for m in order], vert=False, tick_labels=list(order))
plt.axvline(97, color=ORANGE, linestyle="--", label="97% target")
plt.xlabel("Accuracy in each of the 10 folds (%)")
plt.title("10-fold cross-validation")
plt.legend()
plt.savefig(RESULTS / "fig05_cv_boxplot.png")
plt.show()

## Part F — Final test on the held-out 20%
### Task 26: Train every model on the full training set

In [ ]:
for name, model in models.items():
    model.fit(X_train, y_train)
print("All models trained.")

### Task 27: A function that computes all metrics

In [ ]:
def all_metrics(model, X_part, y_part):
    probability = model.predict_proba(X_part)[:, 1]
    predicted = (probability >= 0.5).astype(int)
    return {
        "accuracy": metrics.accuracy_score(y_part, predicted) * 100,
        "balanced_accuracy": metrics.balanced_accuracy_score(y_part, predicted) * 100,
        "precision": metrics.precision_score(y_part, predicted) * 100,
        "recall": metrics.recall_score(y_part, predicted) * 100,
        "specificity": metrics.recall_score(y_part, predicted, pos_label=0) * 100,
        "f1": metrics.f1_score(y_part, predicted) * 100,
        "mcc": metrics.matthews_corrcoef(y_part, predicted),
        "roc_auc": metrics.roc_auc_score(y_part, probability),
        "brier": metrics.brier_score_loss(y_part, probability),
    }

### Task 28: Test-set results table
এটাই মূল ফলাফলের টেবিল।

In [ ]:
test_table = pd.DataFrame({name: all_metrics(model, X_test, y_test) for name, model in models.items()}).T
test_table = test_table.sort_values("accuracy", ascending=False)
test_table.round(3).to_csv(RESULTS / "test_results.csv")
test_table.round(3)

### Task 29: Figure — test accuracy of every model

In [ ]:
ordered = test_table["accuracy"].sort_values()
colours = [ORANGE if m == HYBRID else BLUE for m in ordered.index]
plt.barh(ordered.index, ordered.values, color=colours)
for i, v in enumerate(ordered.values):
    plt.text(v + 0.2, i, f"{v:.2f}%", va="center")
plt.axvline(97, color=GREY, linestyle="--")
plt.xlim(60, 103)
plt.xlabel("Test accuracy (%)")
plt.title("Model comparison on the held-out test set (dashed = 97%)")
plt.savefig(RESULTS / "fig06_test_accuracy.png")
plt.show()

### Task 30: Overfitting check — train accuracy vs test accuracy
ট্রেন আর টেস্টের ফারাক বেশি হলে মডেল মুখস্থ করছে।

In [ ]:
gap_table = pd.DataFrame({
    "train_accuracy": {n: all_metrics(m, X_train, y_train)["accuracy"] for n, m in models.items()},
    "test_accuracy": test_table["accuracy"]})
gap_table["gap"] = gap_table["train_accuracy"] - gap_table["test_accuracy"]
gap_table.round(2).to_csv(RESULTS / "train_vs_test.csv")

gap_table[["train_accuracy", "test_accuracy"]].plot.barh(color=[GREY, BLUE], figsize=(7, 4))
plt.xlim(60, 102)
plt.xlabel("Accuracy (%)")
plt.title("Train vs test accuracy")
plt.savefig(RESULTS / "fig07_train_vs_test.png")
plt.show()
gap_table.round(2)

### Task 31: Figure — ROC curves

In [ ]:
for name, model in models.items():
    fpr, tpr, _ = metrics.roc_curve(y_test, model.predict_proba(X_test)[:, 1])
    plt.plot(fpr, tpr, linewidth=2.5 if name == HYBRID else 1.2,
             label=f"{name} (AUC {test_table.loc[name, 'roc_auc']:.3f})")
plt.plot([0, 1], [0, 1], color=GREY, linestyle="--")
plt.xlabel("False positive rate")
plt.ylabel("True positive rate")
plt.title("ROC curves on the test set")
plt.legend(fontsize=8)
plt.savefig(RESULTS / "fig08_roc_curves.png")
plt.show()

### Task 32: Figure — confusion matrix of every model

In [ ]:
fig, axes = plt.subplots(2, 4, figsize=(14, 6.5))
for ax, (name, model) in zip(axes.ravel(), models.items()):
    matrix = metrics.confusion_matrix(y_test, model.predict(X_test))
    ax.imshow(matrix, cmap="Blues")
    for i in range(2):
        for j in range(2):
            ax.text(j, i, matrix[i, j], ha="center", va="center", fontsize=14,
                    color="white" if matrix[i, j] > matrix.max() / 2 else "black")
    ax.set_xticks([0, 1], ["Not anxious", "Anxious"])
    ax.set_yticks([0, 1], ["Not anxious", "Anxious"], rotation=90, va="center")
    ax.set_xlabel("Predicted")
    ax.set_ylabel("Actual")
    ax.set_title(name, fontsize=10)
axes.ravel()[-1].axis("off")
fig.tight_layout()
fig.savefig(RESULTS / "fig09_confusion_matrices.png")
plt.show()

### Task 33: Figure — calibration of the Hybrid Stack
মডেল ৮০% বললে সত্যিই ৮০% ক্ষেত্রে anxious হয় কি না।

In [ ]:
hybrid = models[HYBRID]
hybrid_probability = hybrid.predict_proba(X_test)[:, 1]
observed, predicted_mean = calibration_curve(y_test, hybrid_probability, n_bins=8, strategy="quantile")
plt.plot(predicted_mean, observed, marker="o", color=BLUE, label=HYBRID)
plt.plot([0, 1], [0, 1], color=GREY, linestyle="--", label="Perfect calibration")
plt.xlabel("Predicted probability")
plt.ylabel("Observed anxious rate")
plt.title(f"Calibration (Brier score {test_table.loc[HYBRID, 'brier']:.3f})")
plt.legend()
plt.savefig(RESULTS / "fig10_calibration.png")
plt.show()

### Task 34: Statistical tests on the 10 cross-validation folds
* **Friedman test:** are the seven models different at all?
* **Wilcoxon signed-rank test:** is the Hybrid Stack different from each other model? (Holm correction for six comparisons)

In [ ]:
friedman = stats.friedmanchisquare(*[cv_scores[m] for m in cv_scores.columns])
print(f"Friedman test: chi2 = {friedman.statistic:.2f}, p = {friedman.pvalue:.2g}")

rows = []
for name in cv_scores.columns.drop(HYBRID):
    difference = cv_scores[HYBRID] - cv_scores[name]
    p = 1.0 if (difference == 0).all() else stats.wilcoxon(cv_scores[HYBRID], cv_scores[name]).pvalue
    rows.append({"hybrid_vs": name, "mean_difference_points": difference.mean() * 100, "p_value": p})
stat_table = pd.DataFrame(rows).sort_values("p_value").reset_index(drop=True)
stat_table["p_holm"] = np.minimum(1, np.maximum.accumulate(stat_table["p_value"] * (len(stat_table) - stat_table.index)))
stat_table.round(4).to_csv(RESULTS / "statistical_tests.csv", index=False)
stat_table.round(4)

### Task 35: McNemar test and bootstrap confidence interval on the test set
টেস্ট সেটে হাইব্রিড বনাম অন্য মডেল, এবং হাইব্রিডের accuracy-র ৯৫% CI।

In [ ]:
hybrid_correct = hybrid.predict(X_test) == y_test
rows = []
for name, model in models.items():
    if name == HYBRID:
        continue
    other_correct = model.predict(X_test) == y_test
    only_hybrid = int((hybrid_correct & ~other_correct).sum())
    only_other = int((~hybrid_correct & other_correct).sum())
    p = stats.binomtest(only_hybrid, only_hybrid + only_other, 0.5).pvalue if only_hybrid + only_other else 1.0
    rows.append({"hybrid_vs": name, "only_hybrid_right": only_hybrid, "only_other_right": only_other, "p_value": round(p, 4)})
mcnemar_table = pd.DataFrame(rows)
mcnemar_table.to_csv(RESULTS / "mcnemar_tests.csv", index=False)

rng = np.random.default_rng(SEED)
boot = [hybrid_correct[rng.integers(0, len(y_test), len(y_test))].mean() * 100 for _ in range(2000)]
ci_low, ci_high = np.percentile(boot, [2.5, 97.5])
print(f"Hybrid Stack test accuracy: {hybrid_correct.mean() * 100:.2f}%  (95% bootstrap CI {ci_low:.2f}-{ci_high:.2f}%)")
mcnemar_table

## Part G — Explainable AI (SHAP and LIME)

| | SHAP | LIME |
|---|---|---|
| Idea | Game theory: share the prediction fairly among the questions | Fit a small linear model around one person |
| Guarantee | Contributions add up exactly to the prediction | No such guarantee |
| Randomness | None here (exact computation) | Uses random samples |

Both methods explain the **Hybrid Stack** for the same 100 test people, so they can be compared fairly.

### Task 36: Prepare the people to explain and the prediction function

In [ ]:
N_EXPLAIN = 100
explain_people = X_test.iloc[:N_EXPLAIN]
background = X_train.sample(30, random_state=SEED)       # reference people for SHAP

def predict_anxious(values):
    # probability of 'anxious' from the Hybrid Stack
    return hybrid.predict_proba(pd.DataFrame(values, columns=FEATURES))[:, 1]

def predict_both(values):
    return hybrid.predict_proba(pd.DataFrame(values, columns=FEATURES))

### Task 37: Compute SHAP values
মাত্র ৯টি প্রশ্ন, তাই SHAP হুবহু (exact) হিসাব করা যায়।

In [ ]:
start = time.time()
shap_explainer = shap.Explainer(predict_anxious, background)
shap_result = shap_explainer(explain_people)
shap_seconds = (time.time() - start) / N_EXPLAIN

shap_values = pd.DataFrame(shap_result.values, columns=FEATURES)
print(f"SHAP: {shap_seconds:.2f} seconds per person")

### Task 38: Figure — SHAP global importance
গড় |SHAP| যত বড়, প্রশ্নটি তত গুরুত্বপূর্ণ।

In [ ]:
shap_importance = shap_values.abs().mean().sort_values()
plt.barh(shap_importance.index, shap_importance.values, color=BLUE)
plt.xlabel("Mean |SHAP value| (change in anxious probability)")
plt.title("SHAP: global importance of each question")
plt.savefig(RESULTS / "fig11_shap_importance.png")
plt.show()

### Task 39: Figure — SHAP effect of each answer (top 4 questions)
কোন উত্তর anxious হওয়ার সম্ভাবনা বাড়ায় (+) আর কোনটা কমায় (−)।

In [ ]:
top4 = shap_importance.index[::-1][:4]
fig, axes = plt.subplots(1, 4, figsize=(15, 3.5), sharey=True)
for ax, feature in zip(axes, top4):
    answer_text = explain_people[feature].map(lambda code: ANSWERS[feature][code]).to_numpy()
    effect = shap_values[feature].groupby(answer_text).mean()
    ax.bar(effect.index, effect.values, color=[ORANGE if v > 0 else BLUE for v in effect.values])
    ax.axhline(0, color="black", linewidth=0.8)
    ax.set_title(feature, fontsize=9)
    ax.tick_params(axis="x", labelsize=8, rotation=15)
axes[0].set_ylabel("Mean SHAP value")
fig.suptitle("SHAP: how each answer moves the prediction (orange = towards anxious)")
fig.tight_layout()
fig.savefig(RESULTS / "fig12_shap_answer_effects.png")
plt.show()

### Task 40: Figure — SHAP explanation of one person

In [ ]:
PERSON = 0
person_answers = {f: ANSWERS[f][explain_people.iloc[PERSON][f]] for f in FEATURES}
labels = [f"{f} = {person_answers[f]}" for f in FEATURES]

def plot_one_person(values, title, filename):
    order = np.argsort(np.abs(values))
    plt.barh([labels[i] for i in order], values[order], color=[ORANGE if v > 0 else BLUE for v in values[order]])
    plt.axvline(0, color="black", linewidth=0.8)
    plt.title(title)
    plt.savefig(RESULTS / filename)
    plt.show()

p = predict_anxious(explain_people.iloc[[PERSON]])[0]
print(f"Predicted anxious probability: {p:.3f}   True label: {y_test[PERSON]}")
print(f"SHAP check: base {shap_result.base_values[PERSON]:.3f} + contributions {shap_values.iloc[PERSON].sum():.3f} = {shap_result.base_values[PERSON] + shap_values.iloc[PERSON].sum():.3f}")
plot_one_person(shap_values.iloc[PERSON].to_numpy(), "SHAP: one person (orange = towards anxious)", "fig13_shap_one_person.png")

### Task 41: Compute LIME explanations
একই ১০০ জনের জন্য LIME।

In [ ]:
lime_explainer = LimeTabularExplainer(
    X_train.to_numpy(), feature_names=FEATURES, class_names=["Not anxious", "Anxious"],
    categorical_features=list(range(len(FEATURES))),
    categorical_names={i: ANSWERS[f] for i, f in enumerate(FEATURES)}, random_state=SEED)

def lime_for(person_row, explainer=lime_explainer):
    explanation = explainer.explain_instance(person_row.to_numpy(), predict_both, num_features=len(FEATURES), num_samples=1000)
    weights = dict(explanation.as_map()[1])                  # {feature number: weight}
    return [weights[i] for i in range(len(FEATURES))]

start = time.time()
lime_values = pd.DataFrame([lime_for(explain_people.iloc[i]) for i in range(N_EXPLAIN)], columns=FEATURES)
lime_seconds = (time.time() - start) / N_EXPLAIN
print(f"LIME: {lime_seconds:.2f} seconds per person")

### Task 42: Figure — LIME global importance

In [ ]:
lime_importance = lime_values.abs().mean().sort_values()
plt.barh(lime_importance.index, lime_importance.values, color=ORANGE)
plt.xlabel("Mean |LIME weight|")
plt.title("LIME: global importance of each question")
plt.savefig(RESULTS / "fig14_lime_importance.png")
plt.show()

### Task 43: Figure — LIME explanation of the same person

In [ ]:
plot_one_person(lime_values.iloc[PERSON].to_numpy(), "LIME: the same person (orange = towards anxious)", "fig15_lime_one_person.png")

### Task 44: SHAP vs LIME (1) — do they rank the questions the same way?
দুই পদ্ধতির importance-কে শতাংশে এনে পাশাপাশি দেখা।

In [ ]:
compare = pd.DataFrame({"SHAP_%": shap_importance / shap_importance.sum() * 100,
                        "LIME_%": lime_importance / lime_importance.sum() * 100}).sort_values("SHAP_%")
compare.round(2).to_csv(RESULTS / "shap_vs_lime_importance.csv")

compare.plot.barh(color=[BLUE, ORANGE], figsize=(7, 4.5))
plt.xlabel("Share of total importance (%)")
plt.title("SHAP vs LIME: global importance")
plt.savefig(RESULTS / "fig16_shap_vs_lime.png")
plt.show()

### Task 45: SHAP vs LIME (2) — agreement
* **Global rank agreement:** Spearman correlation of the two importance rankings.
* **Per-person agreement:** for every person, do both methods pick the same top question, the same top-3, the same direction (+/−)?

In [ ]:
global_spearman = stats.spearmanr(compare["SHAP_%"], compare["LIME_%"]).statistic

S, L = shap_values.to_numpy(), lime_values.to_numpy()
person_spearman = np.mean([stats.spearmanr(np.abs(S[i]), np.abs(L[i])).statistic for i in range(N_EXPLAIN)])
same_top1 = np.mean(np.abs(S).argmax(axis=1) == np.abs(L).argmax(axis=1)) * 100
top3_overlap = np.mean([len(set(np.argsort(-np.abs(S[i]))[:3]) & set(np.argsort(-np.abs(L[i]))[:3])) / 3
                        for i in range(N_EXPLAIN)]) * 100
same_sign = np.mean(np.sign(S) == np.sign(L)) * 100

print(f"Global Spearman rank correlation : {global_spearman:.3f}")
print(f"Mean per-person Spearman         : {person_spearman:.3f}")
print(f"Same top question                : {same_top1:.1f}% of people")
print(f"Top-3 overlap                    : {top3_overlap:.1f}%")
print(f"Same direction (+/-)             : {same_sign:.1f}% of all contributions")

### Task 46: SHAP vs LIME (3) — faithfulness (deletion test)
ধারণা: ব্যাখ্যা যদি সত্যি হয়, তবে "সবচেয়ে গুরুত্বপূর্ণ" প্রশ্নগুলোর উত্তর মুছে দিলে prediction অনেক বদলাবে।
We "remove" the top-k questions of a person (replace them with answers of the 30 reference people) and measure how much the probability moves. A random choice of questions is the baseline.

In [ ]:
def change_after_removing(person_row, feature_numbers):
    changed = pd.concat([person_row.to_frame().T] * len(background), ignore_index=True)
    for j in feature_numbers:
        changed[FEATURES[j]] = background[FEATURES[j]].to_numpy()
    return abs(predict_anxious(person_row.to_frame().T)[0] - predict_anxious(changed).mean())

rng = np.random.default_rng(SEED)
rows = []
for k in [1, 2, 3]:
    by_shap, by_lime, by_random = [], [], []
    for i in range(N_EXPLAIN):
        person_row = explain_people.iloc[i]
        by_shap.append(change_after_removing(person_row, np.argsort(-np.abs(S[i]))[:k]))
        by_lime.append(change_after_removing(person_row, np.argsort(-np.abs(L[i]))[:k]))
        by_random.append(change_after_removing(person_row, rng.choice(len(FEATURES), k, replace=False)))
    rows.append({"questions_removed": k, "SHAP": np.mean(by_shap), "LIME": np.mean(by_lime), "Random": np.mean(by_random)})
faithfulness = pd.DataFrame(rows).set_index("questions_removed")
faithfulness.round(4).to_csv(RESULTS / "shap_vs_lime_faithfulness.csv")

faithfulness.plot.bar(color=[BLUE, ORANGE, GREY], rot=0)
plt.ylabel("Mean change in predicted probability")
plt.xlabel("Number of top questions removed")
plt.title("Faithfulness: higher = the explanation found what really matters")
plt.savefig(RESULTS / "fig17_faithfulness.png")
plt.show()
faithfulness.round(4)

### Task 47: SHAP vs LIME (4) — stability
একই মানুষের জন্য দুইবার চালালে কি একই ব্যাখ্যা আসে?
LIME is re-run with a different random seed; SHAP is re-run with different reference people. 1.0 = identical ranking. (20 people.)

In [ ]:
N_STABLE = 20
lime_explainer_2 = LimeTabularExplainer(
    X_train.to_numpy(), feature_names=FEATURES, class_names=["Not anxious", "Anxious"],
    categorical_features=list(range(len(FEATURES))),
    categorical_names={i: ANSWERS[f] for i, f in enumerate(FEATURES)}, random_state=SEED + 1)
lime_again = np.array([lime_for(explain_people.iloc[i], lime_explainer_2) for i in range(N_STABLE)])

shap_again = shap.Explainer(predict_anxious, X_train.sample(30, random_state=SEED + 1))(explain_people.iloc[:N_STABLE]).values

lime_stability = np.mean([stats.spearmanr(L[i], lime_again[i]).statistic for i in range(N_STABLE)])
shap_stability = np.mean([stats.spearmanr(S[i], shap_again[i]).statistic for i in range(N_STABLE)])
print(f"SHAP stability (new reference people): {shap_stability:.3f}")
print(f"LIME stability (new random seed)     : {lime_stability:.3f}")

### Task 48: SHAP vs LIME — summary table

In [ ]:
xai_summary = pd.DataFrame({
    "SHAP": {"Top question": shap_importance.index[-1],
             "Top-3 questions": ", ".join(shap_importance.index[::-1][:3]),
             "Faithfulness (top-1 removed)": round(faithfulness.loc[1, "SHAP"], 4),
             "Faithfulness (top-3 removed)": round(faithfulness.loc[3, "SHAP"], 4),
             "Stability (Spearman)": round(shap_stability, 3),
             "Seconds per person": round(shap_seconds, 2),
             "Adds up to the prediction": "Yes"},
    "LIME": {"Top question": lime_importance.index[-1],
             "Top-3 questions": ", ".join(lime_importance.index[::-1][:3]),
             "Faithfulness (top-1 removed)": round(faithfulness.loc[1, "LIME"], 4),
             "Faithfulness (top-3 removed)": round(faithfulness.loc[3, "LIME"], 4),
             "Stability (Spearman)": round(lime_stability, 3),
             "Seconds per person": round(lime_seconds, 2),
             "Adds up to the prediction": "No"}})
agreement = pd.Series({"Global Spearman": round(global_spearman, 3), "Per-person Spearman": round(person_spearman, 3),
                       "Same top question (%)": round(same_top1, 1), "Top-3 overlap (%)": round(top3_overlap, 1),
                       "Same direction (%)": round(same_sign, 1)}, name="value")
xai_summary.to_csv(RESULTS / "shap_vs_lime_summary.csv")
agreement.to_csv(RESULTS / "shap_vs_lime_agreement.csv")
print(agreement.to_string())
xai_summary

### Task 49: Do the seven models agree on what is important?
Permutation importance: একটি প্রশ্নের উত্তর এলোমেলো করে দিলে accuracy কত কমে। সব মডেলের জন্য একই পদ্ধতি, তাই তুলনা করা যায়।

In [ ]:
importance_by_model = pd.DataFrame({
    name: permutation_importance(model, X_test, y_test, n_repeats=5, random_state=SEED, scoring="accuracy").importances_mean
    for name, model in models.items()}, index=FEATURES) * 100
importance_by_model = importance_by_model.loc[importance_by_model[HYBRID].sort_values(ascending=False).index]
importance_by_model.round(2).to_csv(RESULTS / "permutation_importance_by_model.csv")

fig, ax = plt.subplots(figsize=(10, 4.5))
image = ax.imshow(importance_by_model.values, cmap="Blues", aspect="auto")
ax.set_xticks(range(len(models)), importance_by_model.columns, rotation=25, ha="right", fontsize=8)
ax.set_yticks(range(len(FEATURES)), importance_by_model.index, fontsize=8)
for i in range(importance_by_model.shape[0]):
    for j in range(importance_by_model.shape[1]):
        value = importance_by_model.values[i, j]
        ax.text(j, i, f"{value:.1f}", ha="center", va="center", fontsize=8,
                color="white" if value > importance_by_model.values.max() / 2 else "black")
fig.colorbar(image, label="Accuracy drop (points)")
ax.set_title("Permutation importance of each question, per model")
fig.savefig(RESULTS / "fig18_importance_by_model.png")
plt.show()

## Part H — Trustworthy-AI extras
### Task 50: Conformal prediction — an honest "I am not sure"
সাধারণ মডেল সবসময় একটাই উত্তর দেয়। Conformal prediction একটি **উত্তরের সেট** দেয়, যার ভেতরে সত্যি উত্তর থাকবে অন্তত (1 − α) সম্ভাবনায়। সেটে দুটো উত্তরই থাকলে মানে মডেল অনিশ্চিত → মানুষ দেখবে।

Steps: (1) get out-of-fold probabilities on the training set, (2) score = 1 − probability of the true class, (3) take the (1 − α) quantile of the scores as the threshold.

In [ ]:
oof_probability = cross_val_predict(hybrid, X_train, y_train, cv=5, method="predict_proba")
calibration_scores = 1 - oof_probability[np.arange(len(y_train)), y_train]
test_probability = hybrid.predict_proba(X_test)

rows = []
for alpha in [0.10, 0.05, 0.01]:
    n = len(calibration_scores)
    threshold = np.quantile(calibration_scores, min(1, np.ceil((n + 1) * (1 - alpha)) / n))
    in_set = (1 - test_probability) <= threshold                   # True where a class is inside the set
    rows.append({"target_coverage_%": (1 - alpha) * 100,
                 "actual_coverage_%": in_set[np.arange(len(y_test)), y_test].mean() * 100,
                 "average_set_size": in_set.sum(axis=1).mean(),
                 "single_answer_%": (in_set.sum(axis=1) == 1).mean() * 100,
                 "uncertain_both_%": (in_set.sum(axis=1) == 2).mean() * 100,
                 "empty_set_%": (in_set.sum(axis=1) == 0).mean() * 100})
conformal_table = pd.DataFrame(rows)
conformal_table.round(2).to_csv(RESULTS / "conformal_prediction.csv", index=False)
conformal_table.round(2)

### Task 51: Figure — conformal coverage
নোট: মডেল খুব আত্মবিশ্বাসী হলে threshold খুব ছোট হয়; তখন কিছু মানুষের জন্য সেট **খালি** হতে পারে (`empty_set_%`)। খালি সেট মানে "এই মানুষটি মডেলের কাছে অস্বাভাবিক" — এদেরও মানুষ দিয়ে দেখানো উচিত।

In [ ]:
positions = np.arange(len(conformal_table))
plt.bar(positions - 0.2, conformal_table["target_coverage_%"], width=0.4, color=GREY, label="Promised coverage")
plt.bar(positions + 0.2, conformal_table["actual_coverage_%"], width=0.4, color=BLUE, label="Coverage on the test set")
plt.xticks(positions, [f"alpha = {1 - c / 100:.2f}" for c in conformal_table["target_coverage_%"]])
plt.ylim(80, 101)
plt.ylabel("Coverage (%)")
plt.title("Conformal prediction: promised vs actual coverage")
plt.legend(loc="lower right")
plt.savefig(RESULTS / "fig19_conformal_coverage.png")
plt.show()

### Task 52: Counterfactual explanation — "what would have to change?"
একজন anxious-predicted মানুষের জন্য খোঁজা হয়: সবচেয়ে কম কয়টি উত্তর বদলালে মডেলের সিদ্ধান্ত উল্টে যায়। Age বদলানো যায় না, তাই বাদ।

In [ ]:
predicted_anxious = np.where(hybrid_probability >= 0.5)[0]
person_number = predicted_anxious[0]
person_row = X_test.iloc[person_number]
changeable = [f for f in FEATURES if f != "Age"]

found = []
for size in [1, 2]:
    for chosen in combinations(changeable, size):
        options = [range(len(ANSWERS[f])) for f in chosen]
        for new_codes in np.array(np.meshgrid(*options)).T.reshape(-1, size):
            if any(new_codes[i] == person_row[f] for i, f in enumerate(chosen)):
                continue
            changed = person_row.copy()
            changed[list(chosen)] = new_codes
            probability = predict_anxious(changed.to_frame().T)[0]
            if probability < 0.5:
                change_text = "; ".join(f"{f}: {ANSWERS[f][person_row[f]]} -> {ANSWERS[f][c]}" for f, c in zip(chosen, new_codes))
                found.append({"answers_changed": size, "change": change_text, "new_probability": round(probability, 3)})
    if found:
        break

print(f"Person {person_number}: anxious probability {hybrid_probability[person_number]:.3f}")
counterfactuals = pd.DataFrame(found).sort_values("new_probability").head(8).reset_index(drop=True)
counterfactuals.to_csv(RESULTS / "counterfactual_example.csv", index=False)
pd.set_option("display.max_colwidth", None)
counterfactuals

### Task 53: Fairness check — accuracy in every age group
মডেল কি সব বয়সের জন্য সমান ভালো?

In [ ]:
age_text = X_test["Age"].map(lambda code: ANSWERS["Age"][code])
fairness = pd.DataFrame({"people": age_text.value_counts(),
                         "accuracy_%": pd.Series(hybrid_correct, index=X_test.index).groupby(age_text).mean() * 100}).sort_index()
fairness.round(2).to_csv(RESULTS / "fairness_by_age.csv")

plt.bar(fairness.index, fairness["accuracy_%"], color=BLUE)
plt.ylim(80, 101)
plt.xlabel("Age group")
plt.ylabel("Hybrid Stack accuracy (%)")
plt.title("Fairness: accuracy by age group")
plt.savefig(RESULTS / "fig20_fairness_by_age.png")
plt.show()
fairness.round(2)

## Part I — Robustness check: duplicate-aware validation
Task 6 showed that many responses share exactly the same answer pattern. In a normal random split, a test person can therefore have an identical "twin" in the training set.

এই অংশে প্রশ্নটা হলো: **যে উত্তর-প্যাটার্ন মডেল আগে কখনো দেখেনি, সেখানে accuracy কত?**
`StratifiedGroupKFold` keeps all copies of one answer pattern in the same fold, so no pattern is in both training and test.

### Task 54: How many test people have a twin in the training set?

In [ ]:
train_patterns = set(map(tuple, X_train.to_numpy()))
has_twin = np.array([tuple(row) in train_patterns for row in X_test.to_numpy()])
print(f"Test people with an identical answer pattern in the training set: {has_twin.sum()} of {len(has_twin)} ({has_twin.mean():.1%})")
for label, mask in [("with a twin", has_twin), ("without a twin", ~has_twin)]:
    print(f"Hybrid Stack accuracy on test people {label:15s}: {hybrid_correct[mask].mean() * 100:.2f}%  (n = {mask.sum()})")

### Task 55: Grouped 5-fold cross-validation (no answer pattern is shared between training and test)

In [ ]:
pattern_id = X.astype(str).agg("-".join, axis=1)
group_folds = StratifiedGroupKFold(n_splits=5, shuffle=True, random_state=SEED)

rows = {}
for name, model in models.items():
    predicted = cross_val_predict(model, X, y, cv=group_folds, groups=pattern_id)
    rows[name] = {"standard_cv_accuracy_%": cv_scores[name].mean() * 100,
                  "grouped_cv_accuracy_%": metrics.accuracy_score(y, predicted) * 100,
                  "grouped_cv_balanced_accuracy_%": metrics.balanced_accuracy_score(y, predicted) * 100}
robustness = pd.DataFrame(rows).T.sort_values("grouped_cv_accuracy_%", ascending=False)
robustness.round(2).to_csv(RESULTS / "robustness_grouped_cv.csv")

robustness[["standard_cv_accuracy_%", "grouped_cv_accuracy_%"]].iloc[::-1].plot.barh(color=[BLUE, ORANGE], figsize=(7, 4.5))
plt.xlim(50, 102)
plt.xlabel("Accuracy (%)")
plt.title("Standard CV vs duplicate-aware (grouped) CV")
plt.legend(["Standard 10-fold CV", "Grouped CV (unseen answer patterns)"], loc="lower right", fontsize=8)
plt.savefig(RESULTS / "fig21_standard_vs_grouped_cv.png")
plt.show()
robustness.round(2)

## Part J — Comparison with published work and final summary
### Task 56: Comparison with earlier papers on the same 1,503-row dataset
আগের পেপারগুলোর সংখ্যা তাদের abstract থেকে নেওয়া (দেখুন `research/01_literature_review.md`)। সবাই সব ১,৫০৩ সারির random split ব্যবহার করেছে, তাই তুলনাটা একই প্রোটোকলে।

In [ ]:
literature = pd.DataFrame([
    {"study": "Nayee et al. 2026 (XGBoost/RF + RFE)", "accuracy_%": 96.99, "explainable_AI": "No", "uncertainty": "No", "statistical_tests": "No"},
    {"study": "Raza et al. 2024 (MDKR meta-learner)", "accuracy_%": 99.0, "explainable_AI": "No", "uncertainty": "No", "statistical_tests": "No"},
    {"study": "Lilhore et al. 2024 (TL + BiLSTM + CNN)", "accuracy_%": 99.6, "explainable_AI": "No", "uncertainty": "No", "statistical_tests": "No"},
    {"study": "This study: CatBoost (best single model)", "accuracy_%": test_table.loc["CatBoost", "accuracy"], "explainable_AI": "-", "uncertainty": "-", "statistical_tests": "Yes"},
    {"study": "This study: " + HYBRID, "accuracy_%": test_table.loc[HYBRID, "accuracy"], "explainable_AI": "SHAP + LIME (compared)", "uncertainty": "Conformal", "statistical_tests": "Yes"},
])
literature.round(2).to_csv(RESULTS / "literature_comparison.csv", index=False)

colours = [ORANGE if "This study" in s else GREY for s in literature["study"]]
plt.barh(literature["study"], literature["accuracy_%"], color=colours)
for i, v in enumerate(literature["accuracy_%"]):
    plt.text(v + 0.1, i, f"{v:.2f}%", va="center")
plt.xlim(90, 101.5)
plt.xlabel("Reported accuracy (%)")
plt.title("This study vs published results on the same dataset")
plt.savefig(RESULTS / "fig22_literature_comparison.png")
plt.show()
literature.round(2)

### Task 57: One table with the headline numbers

In [ ]:
summary = pd.Series({
    "Best model on the test set": test_table.index[0],
    "Hybrid Stack test accuracy (%)": round(test_table.loc[HYBRID, "accuracy"], 2),
    "Hybrid Stack 95% CI (%)": f"{ci_low:.2f}-{ci_high:.2f}",
    "Hybrid Stack F1 (%)": round(test_table.loc[HYBRID, "f1"], 2),
    "Hybrid Stack ROC-AUC": round(test_table.loc[HYBRID, "roc_auc"], 4),
    "Hybrid Stack 10-fold CV accuracy (%)": round(cv_scores[HYBRID].mean() * 100, 2),
    "Hybrid Stack grouped-CV accuracy (%)": round(robustness.loc[HYBRID, "grouped_cv_accuracy_%"], 2),
    "Top question (SHAP)": shap_importance.index[-1],
    "Top question (LIME)": lime_importance.index[-1],
    "SHAP-LIME global Spearman": round(global_spearman, 3),
    "Conformal coverage at 95% target (%)": round(conformal_table.loc[1, "actual_coverage_%"], 2),
}, name="value")
summary.to_csv(RESULTS / "final_summary.csv")
summary.to_frame()